# Day 17: Pure Python RAG Pipeline

Welcome to Day 17 of your AI Engineering journey! Today, we demystify the core mechanism of vector databases by building a Retrieval-Augmented Generation (RAG) pipeline from scratch in pure Python.

## Core Theory (Just-in-Time)

**Why build RAG from scratch?**
As an engineer transitioning to AI, you might see vector databases (like Qdrant or Pinecone) as black boxes. Underneath the hood, retrieval is largely about calculating the distance between two vectors. A RAG pipeline simply:
1. Converts documents into embeddings (dense numerical vectors).
2. Converts a user query into an embedding.
3. Computes a similarity metric (e.g., Cosine Similarity) between the query and documents.
4. Retrieves the top-K most similar documents and injects their text into an LLM prompt.

By building this manually, you understand the exact mechanics of latency, context windows, and vector math before we abstract it away with dedicated vector databases.

**AI Security Implications:**
- **PII / Data Leakage:** When building a custom retrieval system, you must ensure that sensitive data isn't inadvertently embedded or retrieved by unauthorized users. Implement strict role-based access control (RBAC) at the document level.
- **Prompt Injection:** Attackers could attempt to upload documents with malicious instructions intended to override the system prompt. Always isolate retrieved context and enforce rigid boundaries in the prompt template.
- **Fallbacks:** In the event the retriever returns irrelevant documents or the LLM fails to generate a response based on context, you should implement graceful degradation (e.g., "I don't have enough context to answer that securely").



## Code Implementation

Let's build a functional RAG pipeline. We'll use LangChain's `OpenAIEmbeddings` to generate vectors and `ChatOpenAI` for generation, but we will write the retrieval math (Cosine Similarity) in pure Python. 

First, let's create a sample dataset of documents in a JSON file.

In [1]:
import json
import os

# 1. Create a local knowledge base
documents = [
    {"id": "doc1", "text": "The company handbook states that employees get 20 days of paid time off per year.", "department": "HR"},
    {"id": "doc2", "text": "The backend microservices are built with Python and FastAPI, deployed on AWS ECS.", "department": "Engineering"},
    {"id": "doc3", "text": "All customer refunds must be processed within 3 business days of approval.", "department": "Support"},
    {"id": "doc4", "text": "Q3 revenue targets are set to 5 million USD, focusing on enterprise sales.", "department": "Sales"}
]

with open("knowledge_base.json", "w") as f:
    json.dump(documents, f, indent=4)

print("Knowledge base created: knowledge_base.json")

Knowledge base created: knowledge_base.json


### 1. Basic Implementation
Here we isolate the core vector similarity concept with minimal boilerplate. We'll use a mocked embedding function just to demonstrate the math.


In [2]:
import math
from typing import List

def cosine_similarity_basic(vec1: List[float], vec2: List[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    norm_a = math.sqrt(sum(a * a for a in vec1))
    norm_b = math.sqrt(sum(b * b for b in vec2))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

# Dummy embeddings for demonstration
query_vec = [1.0, 0.0, 0.5]
doc_vec = [0.9, 0.1, 0.6]

score = cosine_similarity_basic(query_vec, doc_vec)
print(f"Basic Cosine Similarity: {score:.4f}")



Basic Cosine Similarity: 0.9881


### 2. Medium Implementation
Emphasizing clean OOP and state management. We encapsulate the documents and the retrieval logic inside a class.


In [3]:
class SimpleRetriever:
    def __init__(self):
        # State management: store documents and their embeddings
        self.knowledge_base = []
        
    def add_document(self, doc_id: str, text: str, embedding: List[float]):
        self.knowledge_base.append({
            "id": doc_id,
            "text": text,
            "embedding": embedding
        })
        
    def retrieve(self, query_embedding: List[float], k: int = 1) -> List[dict]:
        scored_docs = []
        for doc in self.knowledge_base:
            score = cosine_similarity_basic(query_embedding, doc["embedding"])
            scored_docs.append((score, doc))
            
        # Sort descending by score
        scored_docs.sort(key=lambda x: x[0], reverse=True)
        return [doc for score, doc in scored_docs[:k]]

# Test the medium implementation
retriever = SimpleRetriever()
retriever.add_document("doc1", "HR Policy: 20 days PTO", [1.0, 0.0, 0.5])
retriever.add_document("doc2", "Tech Stack: Python", [0.0, 1.0, 0.0])

results = retriever.retrieve([0.9, 0.1, 0.6], k=1)
print(f"Medium Retrieval Result: {results[0]['text']}")



Medium Retrieval Result: HR Policy: 20 days PTO


### 3. Advanced Implementation
Production-grade implementation featuring strict type hinting, error handling, exact imports, and AI security best practices (e.g., graceful fallback).


In [4]:
import json
import math
import os
import logging
from typing import List, Dict, Any, Tuple
from pydantic import BaseModel, Field
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

logger = logging.getLogger(__name__)

class Document(BaseModel):
    id: str
    text: str
    department: str
    embedding: List[float] = Field(default_factory=list)

def calculate_cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    """
    Calculates the cosine similarity between two vectors in pure Python.
    Cosine similarity = (A dot B) / (||A|| * ||B||)
    """
    if len(vec1) != len(vec2):
        raise ValueError("Vectors must be of the same length")
    
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    norm_a = math.sqrt(sum(a * a for a in vec1))
    norm_b = math.sqrt(sum(b * b for b in vec2))
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    return dot_product / (norm_a * norm_b)

def load_and_embed_documents(filepath: str, embedder: OpenAIEmbeddings) -> List[Document]:
    """
    Loads documents from a JSON file and generates embeddings for their text.
    """
    with open(filepath, "r") as f:
        raw_docs = json.load(f)
    
    documents = [Document(**doc) for doc in raw_docs]
    
    # In production, batch embedding is crucial for performance.
    texts = [doc.text for doc in documents]
    embeddings = embedder.embed_documents(texts)
    
    for doc, emb in zip(documents, embeddings):
        doc.embedding = emb
        
    return documents

def retrieve_top_k(query_embedding: List[float], documents: List[Document], k: int = 2) -> List[Tuple[Document, float]]:
    """
    Retrieves the top K documents most similar to the query embedding.
    """
    scored_docs = []
    for doc in documents:
        score = calculate_cosine_similarity(query_embedding, doc.embedding)
        scored_docs.append((doc, score))
    
    # Sort by score descending
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return scored_docs[:k]

def generate_answer(query: str, retrieved_docs: List[Document], llm: ChatOpenAI) -> str:
    """
    Constructs a prompt with the retrieved context and calls the LLM to generate an answer.
    """
    context_text = "\n".join([f"- {doc.text} (Source: {doc.department})" for doc in retrieved_docs])
    
    prompt = f"""You are a helpful company assistant. Answer the user's query strictly based on the provided context.
    If the answer is not in the context, output exactly: 'FALLBACK: I do not have enough context to securely answer this query.'
    
    Context:
    {context_text}
    
    User Query: {query}
    """
    
    try:
        response = llm.invoke([
            SystemMessage(content="You are an AI assistant powered by Retrieval-Augmented Generation."),
            HumanMessage(content=prompt)
        ])
        return str(response.content)
    except Exception as e:
        logger.error(f"LLM generation failed: {e}")
        return "FALLBACK: A system error occurred while generating the response."

# --- Execution Pipeline ---
if __name__ == "__main__":
    if "OPENAI_API_KEY" not in os.environ:
        print("Skipping Advanced RAG execution: OPENAI_API_KEY not found in environment.")
    else:
        print("Initializing Pipeline...")
        try:
            embedder = OpenAIEmbeddings(model="text-embedding-3-small")
            llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
            
            print("Loading and embedding documents...")
            docs = load_and_embed_documents("knowledge_base.json", embedder)
            
            user_query = "How many days of PTO do I get?"
            print(f"\nUser Query: {user_query}")
            
            query_emb = embedder.embed_query(user_query)
            top_docs_with_scores = retrieve_top_k(query_emb, docs, k=1)
            
            top_docs = [doc for doc, score in top_docs_with_scores]
            print("\nRetrieved Context:")
            for doc, score in top_docs_with_scores:
                print(f"[{score:.4f}] {doc.text}")
                
            answer = generate_answer(user_query, top_docs, llm)
            print(f"\nLLM Answer:\n{answer}")
        except Exception as e:
            print(f"Execution failed due to missing API configurations or network error: {e}")



Initializing Pipeline...


Loading and embedding documents...


Execution failed due to missing API configurations or network error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


## Common Pitfalls in Production

1.  **Embedding Bottlenecks:** In this manual example, we load and embed documents every time the script runs. In production, this is extremely expensive and slow. You must compute embeddings once, store them in a Vector DB (like Qdrant), and only embed the user's query at runtime.
2.  **Context Overflow:** Retrieving too many documents (high `k`) can exceed the LLM's context window or dilute the signal (the "Lost in the Middle" phenomenon). Keep context concise.
3.  **Missing Metadata Filtering:** Semantic search (cosine similarity) alone is often not enough. If a user asks "What is the HR policy?", but your highest semantic match is an engineering doc that happens to use similar words, the RAG fails. You need hard metadata filters *before* doing vector math.

## Practical Lab / Homework

**Task:** Implement Pre-filtering (Metadata filtering)

In production, we often filter vectors by metadata *before* running the expensive similarity calculation over millions of records.

**Requirements:**
Write a new retrieval function, `retrieve_top_k_with_filter`, that takes an optional `department_filter: str` argument. If provided, it should only calculate cosine similarity and return documents that belong to that specific department.

Below is the complete, working solution implementing this requirement using strict typing and no stubs.
**Bonus:** Record a brief 2-minute async video walkthrough explaining your design decisions and how you managed the filter state.


In [5]:
from typing import List, Tuple, Optional
def retrieve_top_k_with_filter(
    query_embedding: List[float], 
    documents: List[Document], 
    k: int = 2,
    department_filter: Optional[str] = None
) -> List[Tuple[Document, float]]:
    """
    Retrieves the top K documents, optionally applying a strict metadata filter 
    on the 'department' field before calculating similarity.
    """
    # Step 1: Pre-filter the documents
    filtered_docs = documents
    if department_filter is not None:
        filtered_docs = [doc for doc in documents if doc.department == department_filter]
        
    # Step 2: Calculate similarities only on the filtered subset
    scored_docs = []
    for doc in filtered_docs:
        score = calculate_cosine_similarity(query_embedding, doc.embedding)
        scored_docs.append((doc, score))
        
    # Step 3: Sort and return top K
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return scored_docs[:k]
# --- Test the Lab Implementation ---
if __name__ == "__main__" and "OPENAI_API_KEY" in os.environ and 'embedder' in locals():
    try:
        # Assuming `docs` and `embedder` are already loaded from the previous cell
        print("\n--- Lab Execution ---")
        lab_query = "What are the Q3 targets?"
        print(f"Query: {lab_query}")
        
        lab_query_emb = embedder.embed_query(lab_query)
        
        # Filter by a department that DOES NOT contain the answer to prove filtering works
        wrong_department = "HR"
        print(f"Applying filter: Department='{wrong_department}'")
        
        filtered_results = retrieve_top_k_with_filter(
            lab_query_emb, 
            docs, 
            k=2, 
            department_filter=wrong_department
        )
        
        if not filtered_results:
            print("No results found matching the filter criteria.")
        else:
            for doc, score in filtered_results:
                print(f"[{score:.4f}] ({doc.department}) {doc.text}")
                
        print("\nApplying filter: Department='Sales'")
        correct_filtered_results = retrieve_top_k_with_filter(
            lab_query_emb, 
            docs, 
            k=2, 
            department_filter="Sales"
        )
        
        for doc, score in correct_filtered_results:
            print(f"[{score:.4f}] ({doc.department}) {doc.text}")
    
    
    except Exception as e:
        print(f"Execution failed due to missing API configurations or network error: {e}")



--- Lab Execution ---
Query: What are the Q3 targets?
Execution failed due to missing API configurations or network error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


## Reference Links

- [LangChain RAG Documentation](https://python.langchain.com/v0.1/docs/use_cases/question_answering/)
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings)
- [OWASP Top 10 for LLM Applications (AI Security)](https://owasp.org/www-project-top-10-for-large-language-model-applications/)

